### 클래스 불균형 처리 방법 6가지를 실제로 적용하는 예시 코드
분류 모델은 아무런 파라미터 튜닝을 거치지 않은 RandomForest 사용   
본격적으로 각 방법들을 사용해보기 앞서, 가상의 클래스 불균형 데이터를 생성하고 성능 평가 함수를 정의   

In [7]:
# 필요한 모듈 임포트
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from imblearn.metrics import geometric_mean_score

In [3]:
# 가상의 데이터셋 생성 (0번 클래스 900개, 1번 클래스 100개)
X_raw, y_raw = make_classification(
    n_samples=1000, 
    n_features=4, 
    weights=[0.9, 0.1], 
    random_state=42
)

# 데이터프레임 형태로 변환
X = pd.DataFrame(X_raw, columns=['Feature1', 'Feature2', 'Feature3', 'Feature4'])
y = pd.Series(y_raw)

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.3, 
    random_state=42, 
    stratify=y  
)

print("--- SMOTE 적용 전 클래스 분포 ---")
print(y.value_counts())

--- SMOTE 적용 전 클래스 분포 ---
0    895
1    105
Name: count, dtype: int64


In [ ]:
# 성능 평가 시 활용할 함수 정의
# precision, recall, f1-score, roc-auc 출력
def model_performance(y_pred, y_test, y_proba):
    """모델의 성능 지표를 확인하고, 화면에 출력한다.

    Args:
        y_pred : 모델이 예측한 값
        y_test : 실제값
        y_proba : 모델이 내부적으로 계산한 1번 클래스일 확률

    Returns:
        None
    """
    # 지표 계산
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    g_mean = geometric_mean_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    # 결과 딕셔너리 생성
    metrics = {
        "Precision": round(precision, 4),
        "Recall   ": round(recall, 4),
        "F1-Score ": round(f1, 4),
        "G-Mean   ": round(g_mean, 4),
        "ROC-AUC  ": round(roc_auc, 4)
    }
    
    # 출력
    for key, val in metrics.items():
        print(f"{key}: {val}")

#### 1. RandomOverSampler
개념 : 소수 클래스의 데이터를 단순 복사해서 다수 클래스와 데이터 개수를 맞추는 방법   
사용법 : `RandomOverSampler()` 함수의 파라미터로 소수 클래스의 비율 지정 가능   
(예 : `sampling_strategy=0.25` -> 소수클래스:다수클래스 비율을 1:4로 맞추게끔 소수 클래스 데이터를 복사)

In [8]:
from imblearn.over_sampling import RandomOverSampler

# RandomOverSampler 사용해서 train data 오버샘플링하기
ros = RandomOverSampler(sampling_strategy=0.25, random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)


# 분류 모델 생성, 학습, 예측
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_ros, y_train_ros)
y_pred_ros = rf_model.predict(X_test)

# 1번 클래스일 확률값 추출 -> ROC-AUC 계산을 위해
y_proba_ros = rf_model.predict_proba(X_test)[:, 1]

In [13]:
# 성능 지표 출력
model_performance(y_pred_ros, y_test, y_proba_ros)

Precision: 0.9545
Recall   : 0.6774
F1-Score : 0.7925
G-Mean   : 0.8215
ROC-AUC  : 0.9309


#### 2. SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE 사용해서 train data 오버샘플링하기
# 여기서부터는 사용하는 함수, train/pred 데이터 변수명만 바뀌고 코드 전반적인 흐름은 같음
smote = SMOTE(sampling_strategy=0.25, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 분류 모델 생성, 학습, 예측
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)
y_pred_smote = rf_model.predict(X_test)

# 1번 클래스일 확률값 추출 -> ROC-AUC 계산을 위해
y_proba_smote = rf_model.predict_proba(X_test)[:, 1]

In [15]:
# 성능 지표 출력
model_performance(y_pred_smote, y_test, y_proba_smote)

Precision: 0.9167
Recall   : 0.7097
F1-Score : 0.8
G-Mean   : 0.8393
ROC-AUC  : 0.931


### 3. SMOTE Tomek

In [19]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

# SMOTE Tomek 사용해서 train data 오버샘플링하기
# 여기서부터는 사용하는 함수, train/pred 데이터 변수명만 바뀌고 코드 전반적인 흐름은 같음
smt = SMOTETomek(sampling_strategy=0.25, random_state=42)
X_train_smt, y_train_smt = smt.fit_resample(X_train, y_train)

# 분류 모델 생성, 학습, 예측
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_smt, y_train_smt)
y_pred_smt = rf_model.predict(X_test)

# 1번 클래스일 확률값 추출 -> ROC-AUC 계산을 위해
y_proba_smt = rf_model.predict_proba(X_test)[:, 1]



In [20]:
# 성능 지표 출력
model_performance(y_pred_smt, y_test, y_proba_smt)

Precision: 0.92
Recall   : 0.7419
F1-Score : 0.8214
G-Mean   : 0.8581
ROC-AUC  : 0.9279


### 4. SMOTEENN

In [21]:
from imblearn.combine import SMOTEENN

# SMOTEENN 사용해서 train data 오버샘플링하기
# 여기서부터는 사용하는 함수, train/pred 데이터 변수명만 바뀌고 코드 전반적인 흐름은 같음
sme = SMOTEENN(sampling_strategy=0.25, random_state=42)
X_train_sme, y_train_sme = sme.fit_resample(X_train, y_train)

# 분류 모델 생성, 학습, 예측
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_sme, y_train_sme)
y_pred_sme = rf_model.predict(X_test)

# 1번 클래스일 확률값 추출 -> ROC-AUC 계산을 위해
y_proba_sme = rf_model.predict_proba(X_test)[:, 1]

In [22]:
model_performance(y_pred_sme, y_test, y_proba_sme)

Precision: 0.9167
Recall   : 0.7097
F1-Score : 0.8
G-Mean   : 0.8393
ROC-AUC  : 0.9002


### 5. Class Weight
데이터 비율에 맞춰 가중치를 자동으로 계산하는 방법, 가중치를 직접 설정하는 방법이 있음

In [24]:
# 방법 1 : 데이터 비율에 맞춰 가중치 자동 계산
# 가중치 계산식 : 1/{(클래스개수)*(클래스비율)} 
cw_model_auto = RandomForestClassifier(class_weight='balanced', random_state=42)
cw_model_auto.fit(X_train, y_train)
y_pred_cw_auto = cw_model_auto.predict(X_test)
y_proba_cw_auto = cw_model_auto.predict_proba(X_test)[:, 1]

# 방법 2 : 가중치 직접 지정 (예: 1번 소수 클래스에 10배 벌점 부여)
weights = {0: 1.0, 1: 10.0}
cw_model_custom = RandomForestClassifier(class_weight=weights, random_state=42)
cw_model_custom.fit(X_train, y_train)
y_pred_cw_custom = cw_model_custom.predict(X_test)
y_proba_cw_custom = cw_model_custom.predict_proba(X_test)[:, 1]

# 성능 지표 출력
print("---가중치 자동 계산 시 성능---")
model_performance(y_pred_cw_auto, y_test, y_proba_cw_auto)
print()
print("---가중치 직접 지정 시 성능---")
model_performance(y_pred_cw_custom, y_test, y_proba_cw_custom)


---가중치 자동 계산 시 성능---
Precision: 0.8571
Recall   : 0.7742
F1-Score : 0.8136
G-Mean   : 0.8733
ROC-AUC  : 0.9361

---가중치 직접 지정 시 성능---
Precision: 0.8214
Recall   : 0.7419
F1-Score : 0.7797
G-Mean   : 0.8533
ROC-AUC  : 0.95


### 6. Focal Loss